In [1]:
import numpy as np

def write_initial_file():
    with open('InitialFile.txt', 'w') as f:
        f.write('# Initialisation file of the particles\n\n')
        
        # Define numbers of each type of particle
        n_monomers_long_polymer = 500
        n_monomers_short_polymer = 0
        n_HP1 = 400
        nucleation_site_size = 10
        n_ghost = 0
        boundary_elements_bond = 1
        index_IL = 191
        index_IR = 311

        total_monomers = n_monomers_long_polymer + n_monomers_short_polymer
        n_chromatin_bonds_long = n_monomers_long_polymer - 1
        n_chromatin_bonds_short = n_monomers_short_polymer - 1

        f.write(f'{total_monomers + n_HP1 + n_ghost}  atoms\n')
        f.write(f'{n_chromatin_bonds_long + n_chromatin_bonds_short + boundary_elements_bond}  bonds\n\n')
        
        f.write('6  atom types\n')
        f.write('1  bond types\n\n')

        f.write('0.0000   50.0000 xlo xhi\n')
        f.write('0.0000   50.0000 ylo yhi\n')
        f.write('0.0000   50.0000 zlo zhi\n\n')

        f.write('Atom Type Labels\n\n')
        f.write('1    A\n')
        f.write('2    U\n')
        f.write('3    M\n')
        f.write('4    Swi6\n')
        f.write('5    Swi6M\n')
        f.write('6    Epe1C\n\n')

        f.write('Bond Type Labels\n\n')
        f.write('1    Normal\n\n')
        
        f.write('Masses\n\n')
        f.write('1    1.00 \n')
        f.write('2    1.00 \n')
        f.write('3    1.00 \n')
        f.write('4    1.00 \n')
        f.write('5    1.00 \n')
        f.write('6    1.00 \n\n')

        # Assign monomer types for each polymer
        monomer_types = []
        types_long_polymer = np.random.choice(["A", "U", "M"], n_monomers_long_polymer)
        types_long_polymer[n_monomers_long_polymer//2:n_monomers_long_polymer//2 + nucleation_site_size] = "M"
        types_short_polymer = ["M"] * n_monomers_short_polymer
        monomer_types.extend(types_long_polymer)
        monomer_types.extend(types_short_polymer)

        f.write('Atoms\n\n')

        # Generate positions for the long polymer in a chain-like configuration
        offset = 0
        bond_length = 0.6  # Adjust this value as necessary
        x, y, z = 5 + offset, 5 + offset, 5 + offset
        f.write(f'  1  1  {monomer_types[0]}  {x}  {y}  {z}\n')
        for i in range(1, n_monomers_long_polymer):
            direction = np.random.choice(['x', 'y', 'z'])
            if direction == 'x':
                x += bond_length
            elif direction == 'y':
                y += bond_length
            elif direction == 'z':
                z += bond_length
            global_idx = i + 1
            f.write(f'  {global_idx}  1  {monomer_types[i]}  {x}  {y}  {z}\n')
            
        if n_chromatin_bonds_short > 0:
            # Generate positions for the short polymer in a similar chain-like configuration
            offset = 15
            x, y, z = 5 + offset, 5 + offset, 5 + offset
            start_idx = n_monomers_long_polymer
            f.write(f'  {start_idx + 1}  2  {monomer_types[start_idx]}  {x}  {y}  {z}\n')
            for i in range(1, n_monomers_short_polymer):
                direction = np.random.choice(['x', 'y', 'z'])
                if direction == 'x':
                    x += bond_length
                elif direction == 'y':
                    y += bond_length
                elif direction == 'z':
                    z += bond_length
                global_idx = start_idx + i + 1
                f.write(f'  {global_idx}  2  {monomer_types[start_idx + i]}  {x}  {y}  {z}\n')

        # Generate positions for HP1
        hp1_positions = np.random.uniform(1, 50, (n_HP1, 3))
        for i, (x, y, z) in enumerate(hp1_positions):
            f.write(f'  {total_monomers + i + 1}  {total_monomers + i + 1}  Swi6  {x}  {y}  {z}\n')

        # Generate positions for ghost particles
        x, y, z = np.linspace(1, 50, 50), np.linspace(1, 50, 50), np.linspace(1, 50, 50)
        x, y, z = np.meshgrid(x, y, z)
        x, y, z = x.flatten(), y.flatten(), z.flatten()
        
        for i in range(n_ghost):
            f.write(f'  {total_monomers + n_HP1 + i + 1}  {total_monomers + n_HP1 + i + 1}  G  {x[i]}  {y[i]}  {z[i]}\n')

        f.write('\nBonds\n\n')
        # Bonds for the long polymer
        for i in range(n_chromatin_bonds_long):
            f.write(f'  {i + 1}  1  {i + 1}  {i + 2}\n')
        
        f.write(f'  {n_chromatin_bonds_long + 1}  1  {index_IL}  {index_IR}\n')
        
        # Bonds for the short polymer
        if n_chromatin_bonds_short > 0:
            start_idx = n_monomers_long_polymer
            for i in range(n_chromatin_bonds_short):
                f.write(f'  {n_chromatin_bonds_long + i + 1}  1  {start_idx + i + 1}  {start_idx + i + 2}\n')

write_initial_file()
